In [1]:
cd Model/

/home/marcos/Escritorio/SERRATIA/Model


/home/marcos/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Top Down Reconstruction Metabolic Model - Serratia liquefaciens Strain UNJFSC 002

In [2]:
import escher # for visualization
from escher import Builder
import cobra # for model simulations
from time import sleep
escher.rc['never_ask_before_quit'] = True

In [3]:
!wget -nc http://bigg.ucsd.edu/static/models/iJO1366.json

model = cobra.io.load_json_model('iJO1366.json')

--2025-04-18 16:09:27--  http://bigg.ucsd.edu/static/models/iJO1366.json
Resolviendo bigg.ucsd.edu (bigg.ucsd.edu)... 169.228.33.117
Conectando con bigg.ucsd.edu (bigg.ucsd.edu)[169.228.33.117]:80... conectado.
Petición HTTP enviada, esperando respuesta... 200 OK
Longitud: 2948407 (2,8M) [application/json]
Guardando como: ‘iJO1366.json’

iJO1366.json        100%[===================>]   2,81M   703KB/s    en 4,6s    

2025-04-18 16:09:32 (621 KB/s) - ‘iJO1366.json’ guardado [2948407/2948407]



# Inspeccionar las Propiedades del Modelo Metabolico - GEMs

In [4]:
num_reactions = len(model.reactions)
num_metabolites = len(model.metabolites)
num_genes = len(model.genes)

print("Model information")
print("Model ID is: {}".format(model.id))
print("Number of reactions: {}".format(num_reactions))
print("Number of metabolites: {}".format(num_metabolites))
print("Number of genes: {}".format(num_genes))
print("Model compartments:")
for k, v in model.compartments.items():
    print(k + '\t' + v)
print("Model objective is:")
print(model.objective)

Model information
Model ID is: iJO1366
Number of reactions: 2583
Number of metabolites: 1805
Number of genes: 1367
Model compartments:
c	cytosol
e	extracellular space
p	periplasm
Model objective is:
Maximize
1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


In [5]:
## Funciones del Modelo
print(model.summary) #(float_format = None, names = True)) #mmol/gDW/h

<bound method Model.summary of <Model iJO1366 at 0x72c5002bdcf0>>


In [7]:
# Biomasa del Modelo - Metabolitos - Rx - Flux - Rangos - C-number - C-Flux
model.summary(fva=0.95) #, float_format = None, names = True)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
ca2_e,EX_ca2_e,0.005113,[0.004858; 0.005113],0,0.00%
cl_e,EX_cl_e,0.005113,[0.004858; 0.005113],0,0.00%
cobalt2_e,EX_cobalt2_e,2.456E-05,[2.333E-05; 2.456E-05],0,0.00%
cu2_e,EX_cu2_e,0.0006965,[0.0006617; 0.0006965],0,0.00%
fe2_e,EX_fe2_e,0.01578,[-4.978; 1000],0,0.00%
glc__D_e,EX_glc__D_e,10,[9.507; 10],6,100.00%
k_e,EX_k_e,0.1918,[0.1822; 0.1918],0,0.00%
mg2_e,EX_mg2_e,0.008522,[0.008096; 0.008522],0,0.00%
mn2_e,EX_mn2_e,0.0006788,[0.0006449; 0.0006788],0,0.00%
mobd_e,EX_mobd_e,0.0001267,[0.0001204; 0.0001271],0,0.00%


### Reacciones del Modelo Metabolico de iJO1366 para Serratia liquefaciens Strain UNJFSC 002 - (Nitrogeno - Fosfato - GABA - IAA)

In [8]:
def buscar_reacciones(modelo, palabra_clave):
    resultados = []
    for rxn in modelo.reactions:
        if palabra_clave.lower() in rxn.name.lower() or palabra_clave.lower() in rxn.id.lower():
            resultados.append((rxn.id, rxn.name))
    return resultados

In [9]:
# Para poder guardar los formatos en "csv" activamos pandas
import pandas as pd

In [11]:
reacciones_nitrogeno = []
for keyword in ['ammonia', 'nitrate', 'urea', 'glutamine', 'glutamate', 'nitrogen']:
    reacciones_nitrogeno += buscar_reacciones(model, keyword)
# Eliminar duplicados
reacciones_nitrogeno = list(set(reacciones_nitrogeno))
# Convertir a DataFrame
df_nitrogeno = pd.DataFrame(reacciones_nitrogeno, columns=["reaction_id", "reaction_name"])
# Exportar a CSV
df_nitrogeno.to_csv("reacciones_nitrogeno.csv", index=False)
# Opcional: imprimir para revisar
print(df_nitrogeno.head())

             reaction_id                                      reaction_name
0                NO3t7pp  Nitrate transport in via nitrite antiport (per...
1               GLNabcpp   L-glutamine transport via ABC system (periplasm)
2  EX_LalaDgluMdapDala_e  L-alanine-D-glutamate-meso-2,6-diaminoheptaned...
3                 GLUtex  L-glutamate transport via diffusion (extracell...
4                  GLU5K                                 Glutamate 5-kinase


In [13]:
reacciones_fosfato = []
for keyword in ['phosphate', 'phosphatase', 'phosphonate', 'phosphorus']:
    reacciones_fosfato += buscar_reacciones(model, keyword)
# Eliminar duplicados
reacciones_fosfato = list(set(reacciones_fosfato))
# Convertir a DataFrame
df_fosfato = pd.DataFrame(reacciones_fosfato, columns=["reaction_id", "reaction_name"])
# Guardar como CSV
df_fosfato.to_csv("reacciones_fosfato.csv", index=False)
# Mostrar ejemplo
print(df_fosfato.head())

  reaction_id                                      reaction_name
0       NTPP4    Nucleoside triphosphate pyrophosphorylase (ctp)
1   PAPA141pp   Phosphatidate phosphatase (periplasmic, n-C14:1)
2        NTP5                    Nucleoside-triphosphatase (CTP)
3   ACPPAT141  Acyl-(acyl carrier protein):phosphate acyltran...
4        F6PP                 D-fructose 6-phosphate phosphatase


In [14]:
reacciones_gaba = buscar_reacciones(model, 'GABA')
# Convertir a DataFrame
df_gaba = pd.DataFrame(reacciones_gaba, columns=["reaction_id", "reaction_name"])
# Exportar a CSV
df_gaba.to_csv("reacciones_gaba.csv", index=False)
# Mostrar primeras filas como verificación
print(df_gaba.head())

  reaction_id                                      reaction_name
0    GGGABADr  Gamma-glutamyl-gamma aminobutyric acid dehydro...
1     GGGABAH   Gamma-glutamyl-gamma-aminobutyric acid hydrolase


In [15]:
reacciones_iaa = []
for keyword in ['indole', 'tryptophan', 'IAA', 'indole-3-acetic']:
    reacciones_iaa += buscar_reacciones(model, keyword)
# Eliminar duplicados
reacciones_iaa = list(set(reacciones_iaa))
# Convertir a DataFrame
df_iaa = pd.DataFrame(reacciones_iaa, columns=["reaction_id", "reaction_name"])
# Guardar como CSV
df_iaa.to_csv("reacciones_iaa.csv", index=False)
# Verificar resultados
print(df_iaa.head())

   reaction_id                                      reaction_name
0     TRPt2rpp  L-tryptophan reversible transport via proton s...
1         IGPS               Indole-3-glycerol-phosphate synthase
2        TRPS1     Tryptophan synthase (indoleglycerol phosphate)
3  EX_indole_e                                    Indole exchange
4   INDOLEt2pp  Indole transport via proton symport, irreversi...


In [20]:
ls

iJO1366.json            reacciones_gaba.csv  reacciones_nitrogeno.csv
reacciones_fosfato.csv  reacciones_iaa.csv


## Creacion del Modelo Metabolico iJO1366

In [26]:
from escher import Builder
import json
import requests

In [35]:
escher.list_available_maps()

[{'organism': 'Saccharomyces cerevisiae',
  'map_name': 'iMM904.Central carbon metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Inositol retinol metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Glycolysis TCA PPP'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Tryptophan metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Carbohydrate metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Amino acid metabolism (partial)'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Nucleotide metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid biosynthesis (saturated)'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Nucleotide and histidine biosynthesis'},
 {'organism': 'Escherichia coli', 'map_name': 'e_coli_core.Core metabolism'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Central metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid beta-oxidation'}

In [36]:
escher.list_available_models()

[{'organism': 'Saccharomyces cerevisiae', 'model_name': 'iMM904'},
 {'organism': 'Homo sapiens', 'model_name': 'RECON1'},
 {'organism': 'Escherichia coli', 'model_name': 'e_coli_core'},
 {'organism': 'Escherichia coli', 'model_name': 'iJO1366'}]

In [37]:
builder = Builder(
    map_name='iJO1366.Central metabolism',
    model_name='iJO1366'
)

In [39]:
builder = Builder(
    height=600,
    map_name=None,
    model_name='iJO1366',
    map_json='https://escher.github.io/1-0-0/6/models/Escherichia%20coli/iJO1366.json',
)
builder

Builder(height=600, never_ask_before_quit=True)

In [40]:
builder = Builder(
    height=600,
    map_name=None,
    model_name='iJO1366',
    map_json='https://escher.github.io/1-0-0/6/maps/Escherichia%20coli/iJO1366.Central%20metabolism.json',
)
builder.save_html('mi_mapitaa.html')

In [41]:
timestep = 0.1
duration = 60 # seconds
min_flux = 0 # minimum flux value
max_flux = 20 # maximum flux value
step = 1 # increment in flux value
val = min_flux
with model:
    for _ in range(int(duration / timestep)):
        model.reactions.EX_o2_e.lower_bound = -val # exchange flux variation
        solution = model.optimize() # solve model
        builder.reaction_data = solution.fluxes # add fluxes to model
        if val <= max_flux:
            val += step # increment step
        else:
            val = min_flux # reset
        sleep(timestep)

In [42]:
builder.save_html('e_coli_Central_map.html')

In [43]:
escher.list_available_maps()

[{'organism': 'Saccharomyces cerevisiae',
  'map_name': 'iMM904.Central carbon metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Inositol retinol metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Glycolysis TCA PPP'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Tryptophan metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Carbohydrate metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Amino acid metabolism (partial)'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Nucleotide metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid biosynthesis (saturated)'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Nucleotide and histidine biosynthesis'},
 {'organism': 'Escherichia coli', 'map_name': 'e_coli_core.Core metabolism'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Central metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid beta-oxidation'}